In [1]:
"""
Week 8 - Function 1 (REVISED)
Key upgrades vs your run:
- Signed log transform for y to handle extreme dynamic range + negatives
- Less restrictive likelihood prior for sigma
- More exploitative decoding to target the "next best" point
- x_next printed to 6 decimals (and dedup uses same rounding)
"""

import math
import numpy as np
from dataclasses import dataclass

import torch
import torch.nn as nn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.optim import Adam


# ------------------------ 1. Base Config ------------------------

RANDOM_SEED = 123
INPUT_DIM = 2
HIDDEN_SIZES = [64, 64]

N_EPOCHS_FEATURE = 2000
LR_FEATURE = 1e-3

N_TUNING_TRIALS = 20
N_FOLDS = 4

# Candidate budget ("max tokens")
MAX_TOKENS = 40000
N_GLOBAL = 26000
N_LOCAL = 14000

# More exploitative decoding to get a clear "next best"
TOP_K = 24
TOP_P = 0.75
TEMPERATURE = 0.12

# 6-decimal policy for output + dedup
DUP_DECIMALS = 6

# Local refinement
N_BEST_ANCHORS = 4
LOCAL_SIGMA = 0.06

# fallbacks
DEFAULT_PRIOR_SCALE = 1.0
DEFAULT_LR_VI = 3e-3
DEFAULT_N_STEPS_VI = 5000
DEFAULT_N_PRED_SAMPLES = 256
DEFAULT_XI = 0.0025


# ------------------------ 2. Data ------------------------

def load_data():
    X_raw = np.array([
        [0.31940389, 0.76295937],
        [0.57432921, 0.87989810],
        [0.73102363, 0.73299988],
        [0.84035342, 0.26473161],
        [0.65011406, 0.68152635],
        [0.41043714, 0.14755430],
        [0.31269116, 0.07872278],
        [0.68341817, 0.86105746],
        [0.08250725, 0.40348751],
        [0.88388983, 0.58225397],
        [0.88389,    0.98389],
        [0.37454,    0.950713],
        [0.382224,   0.951319],
        [0.782778,   0.793329],
        [0.030500,   0.037300],
        [0.646168,   0.172681],
        [0.546683,   0.562315]
    ], dtype=np.float64)

    y_raw = np.array([
        1.32267704e-79,
        1.03307824e-46,
        7.71087511e-16,
        3.34177101e-124,
        -3.60606264e-03,
        -2.15924904e-54,
        -2.08909327e-91,
        2.53500115e-40,
        3.60677119e-81,
        6.22985647e-48,
        9.59033053e-135,
        -1.56227724e-117,
        -4.77166224e-115,
        7.535209723645751e-36,
        1.6357533426693436e-209,
        6.327028545366271e-79,
        3.45699404834516e-08
    ], dtype=np.float64)

    assert X_raw.shape[1] == INPUT_DIM
    return X_raw, y_raw


# ------------------------ 3. Signed-log transform for y ------------------------

def signed_log_transform(y: np.ndarray, s: float):
    return np.sign(y) * np.log1p(np.abs(y) / s)

def signed_log_inverse(y_t: np.ndarray, s: float):
    return np.sign(y_t) * s * np.expm1(np.abs(y_t))

def compute_scale_s(y: np.ndarray):
    # robust scale for transform (avoid being dominated by the -0.0036 point)
    med = float(np.median(np.abs(y)))
    return max(med, 1e-12)


# ------------------------ 4. Feature extractor (deterministic) ------------------------

class FeatureExtractor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            prev_dim = h
        self.net = nn.Sequential(*layers)
        self.output_dim = prev_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DeterministicRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes):
        super().__init__()
        self.feature_extractor = FeatureExtractor(input_dim, hidden_sizes)
        self.head = nn.Linear(self.feature_extractor.output_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.feature_extractor(x)
        return self.head(feats).squeeze(-1)


def train_feature_extractor(X: torch.Tensor, y: torch.Tensor) -> FeatureExtractor:
    torch.manual_seed(RANDOM_SEED)
    model = DeterministicRegressor(INPUT_DIM, HIDDEN_SIZES)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_FEATURE)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(N_EPOCHS_FEATURE):
        optimizer.zero_grad()
        preds = model(X)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    feature_extractor = FeatureExtractor(INPUT_DIM, HIDDEN_SIZES)
    feature_extractor.load_state_dict(model.feature_extractor.state_dict())
    return feature_extractor


# ------------------------ 5. Bayesian linear head (Pyro) ------------------------

@dataclass
class BayesianHeadConfig:
    prior_scale: float


def make_bayesian_model(feature_extractor: FeatureExtractor, cfg: BayesianHeadConfig):
    def model(x, y=None):
        pyro.module("feature_extractor", feature_extractor, update_module_params=False)
        feats = feature_extractor(x)
        H = feats.size(-1)

        weight = pyro.sample(
            "weight",
            dist.Normal(x.new_zeros(H), cfg.prior_scale * x.new_ones(H)).to_event(1)
        )
        bias = pyro.sample("bias", dist.Normal(x.new_tensor(0.0), cfg.prior_scale))

        # IMPORTANT CHANGE: much less tight noise prior (stabilizes VI + CV loglik)
        sigma = pyro.sample("sigma", dist.HalfCauchy(x.new_tensor(1.0)))

        mean = (feats * weight).sum(dim=-1) + bias
        with pyro.plate("data", x.size(0)):
            pyro.sample("obs", dist.Normal(mean, sigma), obs=y)

    return model


def train_bayesian_head(model, X: torch.Tensor, y: torch.Tensor, lr_vi: float, n_steps_vi: int):
    pyro.clear_param_store()
    guide = AutoDiagonalNormal(model)
    optimizer = Adam({"lr": lr_vi})
    svi = SVI(model, guide, optimizer, loss=Trace_ELBO())
    for _ in range(n_steps_vi):
        svi.step(X, y)
    return guide


# ------------------------ 6. Acquisition ------------------------

def normal_cdf(x: torch.Tensor) -> torch.Tensor:
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

def normal_pdf(x: torch.Tensor) -> torch.Tensor:
    return (1.0 / math.sqrt(2.0 * math.pi)) * torch.exp(-0.5 * x**2)

def compute_ei_and_pi(samples: torch.Tensor, best_y: float, xi: float):
    mean = samples.mean(dim=0)
    std = samples.std(dim=0) + 1e-9
    gamma = (mean - best_y - xi) / std
    ei = (mean - best_y - xi) * normal_cdf(gamma) + std * normal_pdf(gamma)
    ei = torch.clamp(ei, min=0.0)
    pi = normal_cdf((mean - best_y - xi) / std)
    return ei, pi, mean, std


# ------------------------ 7. Tuning (Random Search + CV) ------------------------

def mc_predictive_mean_std(model, guide, X: torch.Tensor, n_samples: int):
    predictive = Predictive(model, guide=guide, num_samples=n_samples, return_sites=("obs",))
    with torch.no_grad():
        pred = predictive(X)["obs"]
    return pred.mean(dim=0), pred.std(dim=0) + 1e-9

def cv_score(feature_extractor: FeatureExtractor,
             X_all: torch.Tensor,
             y_all: torch.Tensor,
             prior_scale: float,
             lr_vi: float,
             n_steps_vi: int,
             n_pred_samples: int):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for train_idx, val_idx in kf.split(X_all.cpu().numpy()):
        X_tr, y_tr = X_all[train_idx], y_all[train_idx]
        X_va, y_va = X_all[val_idx], y_all[val_idx]

        cfg = BayesianHeadConfig(prior_scale=prior_scale)
        model = make_bayesian_model(feature_extractor, cfg)
        guide = train_bayesian_head(model, X_tr, y_tr, lr_vi=lr_vi, n_steps_vi=n_steps_vi)

        mu, sd = mc_predictive_mean_std(model, guide, X_va, n_samples=n_pred_samples)
        ll = (-0.5 * torch.log(2 * math.pi * sd**2) - 0.5 * ((y_va - mu) ** 2) / (sd**2)).mean()
        scores.append(ll.item())
    return float(np.mean(scores))

def random_log_uniform(rng, low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def tune_hyperparameters(feature_extractor: FeatureExtractor, X_all: torch.Tensor, y_all: torch.Tensor):
    rng = np.random.default_rng(RANDOM_SEED)
    steps_choices = [2000, 3500, 5000, 6500]
    pred_choices = [256, 384, 512]

    best = {
        "score": -1e18,
        "prior_scale": DEFAULT_PRIOR_SCALE,
        "lr_vi": DEFAULT_LR_VI,
        "n_steps_vi": DEFAULT_N_STEPS_VI,
        "n_pred_samples": DEFAULT_N_PRED_SAMPLES,
        "xi": DEFAULT_XI,
    }

    for _ in range(N_TUNING_TRIALS):
        prior_scale = random_log_uniform(rng, 0.08, 4.0)
        lr_vi = random_log_uniform(rng, 8e-4, 8e-3)
        n_steps_vi = int(rng.choice(steps_choices))
        n_pred_samples = int(rng.choice(pred_choices))
        xi = random_log_uniform(rng, 5e-5, 8e-3)

        try:
            score = cv_score(feature_extractor, X_all, y_all,
                             prior_scale=prior_scale,
                             lr_vi=lr_vi,
                             n_steps_vi=n_steps_vi,
                             n_pred_samples=n_pred_samples)
        except Exception:
            continue

        if score > best["score"]:
            best.update({
                "score": score,
                "prior_scale": prior_scale,
                "lr_vi": lr_vi,
                "n_steps_vi": n_steps_vi,
                "n_pred_samples": n_pred_samples,
                "xi": xi,
            })
    return best


# ------------------------ 8. Week 8 candidate policy + decoding ------------------------

def _rounded_key(x_np: np.ndarray, decimals: int = DUP_DECIMALS):
    return tuple(np.round(x_np.astype(np.float64), decimals=decimals).tolist())

def build_candidates_week8(X_train_raw: np.ndarray, y_train_raw: np.ndarray, device: torch.device):
    torch.manual_seed(RANDOM_SEED)

    sob = torch.quasirandom.SobolEngine(dimension=INPUT_DIM, scramble=True, seed=RANDOM_SEED)
    Xg = sob.draw(N_GLOBAL).to(device)

    best_idx = np.argsort(-y_train_raw)[:min(N_BEST_ANCHORS, len(y_train_raw))]
    anchors = torch.from_numpy(X_train_raw[best_idx].astype(np.float32)).to(device)

    idx = torch.randint(low=0, high=anchors.size(0), size=(N_LOCAL,), device=device)
    noise = LOCAL_SIGMA * torch.randn((N_LOCAL, INPUT_DIM), device=device)
    Xl = torch.clamp(anchors[idx] + noise, 0.0, 1.0)

    Xcand = torch.cat([Xg, Xl], dim=0)
    if Xcand.size(0) > MAX_TOKENS:
        Xcand = Xcand[:MAX_TOKENS]

    train_keys = set(_rounded_key(x) for x in X_train_raw)
    seen = set(train_keys)
    keep = []

    Xcand_np = Xcand.detach().cpu().numpy()
    for i in range(Xcand_np.shape[0]):
        k = _rounded_key(Xcand_np[i])
        if k in seen:
            continue
        seen.add(k)
        keep.append(i)

    if len(keep) == 0:
        Xcand = torch.rand((min(2048, MAX_TOKENS), INPUT_DIM), device=device)
    else:
        Xcand = Xcand[torch.tensor(keep, device=device)]
    return Xcand

def decode_pick_from_top(ei: torch.Tensor, temperature: float, top_k: int, top_p: float):
    ei = torch.clamp(ei, min=0.0)
    if ei.numel() == 0:
        return None

    k = min(int(top_k), int(ei.numel()))
    top_vals, top_idx = torch.topk(ei, k=k)

    if float(top_vals.sum().item()) <= 1e-12:
        return int(top_idx[0].item())

    probs = top_vals / top_vals.sum()
    sorted_probs, order = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=0)
    m = int(torch.searchsorted(cum, torch.tensor(top_p, device=ei.device)).item()) + 1
    m = max(1, min(m, sorted_probs.numel()))

    nucleus_order = order[:m]
    nucleus_probs = sorted_probs[:m]

    T = max(1e-6, float(temperature))
    logits = torch.log(nucleus_probs + 1e-12) / T
    weights = torch.softmax(logits, dim=0)

    j = torch.multinomial(weights, num_samples=1).item()
    chosen_top_index = int(nucleus_order[j].item())
    chosen_global_index = int(top_idx[chosen_top_index].item())
    return chosen_global_index


def propose_next_point_week8(model, guide,
                             X_train_scaled: torch.Tensor,
                             y_train_scaled: torch.Tensor,
                             X_train_raw_np: np.ndarray,
                             y_train_raw_np: np.ndarray,
                             scaler_y_transformed: StandardScaler,
                             y_scale_s: float,
                             xi: float,
                             n_pred_samples: int):
    device = X_train_scaled.device
    X_candidates = build_candidates_week8(X_train_raw_np, y_train_raw_np, device=device)

    predictive = Predictive(model, guide=guide, num_samples=n_pred_samples, return_sites=("obs",))
    with torch.no_grad():
        samples_scaled = predictive(X_candidates)["obs"]  # (S, N)

    best_y_scaled = float(y_train_scaled.max().item())
    ei, pi, mean_scaled, std_scaled = compute_ei_and_pi(samples_scaled, best_y_scaled, xi=xi)

    idx_decoded = decode_pick_from_top(ei, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P)
    idx_best = int(torch.argmax(ei).item()) if idx_decoded is None else idx_decoded

    next_x = X_candidates[idx_best].detach().cpu().numpy()

    # Inverse transform: scaled -> transformed y' -> raw y
    mean_scaled_np = mean_scaled.detach().cpu().numpy()
    std_scaled_np = std_scaled.detach().cpu().numpy()

    mean_trans = scaler_y_transformed.inverse_transform(mean_scaled_np.reshape(-1, 1)).squeeze(-1)
    mean_raw = signed_log_inverse(mean_trans, y_scale_s)

    # std: approximate by mapping std in transformed space back to raw around mean
    std_trans = std_scaled_np * scaler_y_transformed.scale_[0]
    std_raw = np.abs(signed_log_inverse(mean_trans + std_trans, y_scale_s) - signed_log_inverse(mean_trans, y_scale_s))

    best_y_trans = float(scaler_y_transformed.inverse_transform(np.array(best_y_scaled).reshape(-1, 1)).squeeze())
    best_y_raw = float(signed_log_inverse(np.array([best_y_trans]), y_scale_s).squeeze())

    topk_vals, topk_idx = torch.topk(ei, k=min(10, ei.numel()))
    top_candidates = []
    for rank, (ei_val, j) in enumerate(zip(topk_vals, topk_idx)):
        j = int(j.item())
        x_j = X_candidates[j].detach().cpu().numpy()
        top_candidates.append({
            "rank": rank + 1,
            "x": x_j,
            "ei_scaled": float(ei_val.item()),
            "pred_mean": float(mean_raw[j]),
            "pred_std": float(std_raw[j]),
            "prob_improvement": float(pi[j].item()),
        })

    info = {
        "next_x": next_x,
        "next_pred_mean": float(mean_raw[idx_best]),
        "next_pred_std": float(std_raw[idx_best]),
        "next_prob_improvement": float(pi[idx_best].item()),
        "best_y_raw": best_y_raw,
        "top_candidates": top_candidates,
        "decode_settings": {
            "temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K,
            "max_tokens": MAX_TOKENS,
            "policy": {"global": N_GLOBAL, "local": N_LOCAL, "local_sigma": LOCAL_SIGMA, "anchors": N_BEST_ANCHORS},
        }
    }
    return next_x, info


# ------------------------ 9. Main ------------------------

def main():
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    X_raw, y_raw = load_data()
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")

    # Signed-log transform for y then standardize
    y_s = compute_scale_s(y_raw)
    y_trans = signed_log_transform(y_raw, y_s)

    scaler_y_trans = StandardScaler()
    y_scaled_np = scaler_y_trans.fit_transform(y_trans.reshape(-1, 1)).astype(np.float32).ravel()

    device = torch.device("cpu")
    X_t = torch.from_numpy(X_raw.astype(np.float32)).to(device)
    y_t = torch.from_numpy(y_scaled_np.astype(np.float32)).to(device)

    print("\n=== Pretraining deterministic feature extractor ===")
    feature_extractor = train_feature_extractor(X_t, y_t)
    feature_extractor.eval()
    print("Feature extractor trained.")

    print("\n=== Hyperparameter tuning (Random Search + 4-fold CV) ===")
    best_hp = tune_hyperparameters(feature_extractor, X_t, y_t)

    print("Best hyperparameters found:")
    print(f"  CV score (avg log-lik): {best_hp['score']:.6f}")
    print(f"  prior_scale:    {best_hp['prior_scale']:.6g}")
    print(f"  lr_vi:          {best_hp['lr_vi']:.6g}")
    print(f"  n_steps_vi:     {best_hp['n_steps_vi']}")
    print(f"  n_pred_samples: {best_hp['n_pred_samples']}")
    print(f"  xi (EI/PI):     {best_hp['xi']:.6g}")

    print("\n=== Training final Bayesian head on ALL data (best HP) ===")
    bayes_cfg = BayesianHeadConfig(prior_scale=best_hp["prior_scale"])
    bayes_model = make_bayesian_model(feature_extractor, bayes_cfg)
    guide = train_bayesian_head(
        bayes_model, X_t, y_t,
        lr_vi=best_hp["lr_vi"],
        n_steps_vi=best_hp["n_steps_vi"]
    )
    print("Final Bayesian head training complete.")

    print("\n=== Proposing next query point (Week 8 improved) ===")
    next_x, info = propose_next_point_week8(
        bayes_model, guide,
        X_t, y_t,
        X_train_raw_np=X_raw,
        y_train_raw_np=y_raw,
        scaler_y_transformed=scaler_y_trans,
        y_scale_s=y_s,
        xi=best_hp["xi"],
        n_pred_samples=best_hp["n_pred_samples"]
    )

    # Print x_next to 6 decimals only
    x6 = np.round(info["next_x"].astype(np.float64), 6)
    print("\n=== CURRENT BEST (from observed data) ===")
    print(f"Best observed y (original scale): {info['best_y_raw']:.6g}")

    print("\n=== PROPOSED NEXT QUERY POINT ===")
    print(f"x_next (6 d.p., in [0,1]^2): [{x6[0]:.6f} {x6[1]:.6f}]")
    print(f"Predicted y at x_next (mean, original scale): {info['next_pred_mean']:.6g}")
    print(f"Predictive std at x_next (original scale): {info['next_pred_std']:.6g}")
    print(f"Probability of improvement over current best: {info['next_prob_improvement'] * 100:.2f}%")

    print("\n=== DECODING SETTINGS (Week 8) ===")
    ds = info["decode_settings"]
    print(f"temperature={ds['temperature']}, top_p={ds['top_p']}, top_k={ds['top_k']}, max_tokens={ds['max_tokens']}")
    pol = ds["policy"]
    print(f"candidate_policy: global={pol['global']} local={pol['local']} anchors={pol['anchors']} local_sigma={pol['local_sigma']}")

    print("\n=== TOP CANDIDATES (by EI, in [0,1]^2) ===")
    for cand in info["top_candidates"]:
        x = np.round(cand["x"].astype(np.float64), 6)
        print(
            f"Rank {cand['rank']:>2d}: x=[{x[0]:.6f} {x[1]:.6f}], "
            f"pred_mean={cand['pred_mean']:.6g}, "
            f"pred_std={cand['pred_std']:.6g}, "
            f"PI={cand['prob_improvement']*100:5.2f}%, "
            f"EI_scaled={cand['ei_scaled']:.4g}"
        )

    print("\nDone.")


if __name__ == "__main__":
    main()


Loaded X_raw shape: (17, 2), y_raw shape: (17,)

=== Pretraining deterministic feature extractor ===
Feature extractor trained.

=== Hyperparameter tuning (Random Search + 4-fold CV) ===
Best hyperparameters found:
  CV score (avg log-lik): -15.597973
  prior_scale:    0.197923
  lr_vi:          0.00117218
  n_steps_vi:     2000
  n_pred_samples: 384
  xi (EI/PI):     0.000962421

=== Training final Bayesian head on ALL data (best HP) ===
Final Bayesian head training complete.

=== Proposing next query point (Week 8 improved) ===

=== CURRENT BEST (from observed data) ===
Best observed y (original scale): 3.45699e-08

=== PROPOSED NEXT QUERY POINT ===
x_next (6 d.p., in [0,1]^2): [0.814946 0.846025]
Predicted y at x_next (mean, original scale): -2.16127e-13
Predictive std at x_next (original scale): 1.1406e-09
Probability of improvement over current best: 7.05%

=== DECODING SETTINGS (Week 8) ===
temperature=0.12, top_p=0.75, top_k=24, max_tokens=40000
candidate_policy: global=26000 lo